# EDA — Paraguay Geodata Overview

Overview of all 14 datasets in `/root/paraguay-geodata/exports/web/data/`

**Source:** https://github.com/Ai-Whisperers/paraguay-geodata
**Size:** 549 MB

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path('/root/paraguay-geodata/exports/web/data')

In [ ]:
from src.paraguay_admin import (
    load_departamentos,
    load_distritos,
    load_tile_index,
    load_catastro_parcels,
    load_indigenous_territories,
)

In [ ]:
# Summary table
summary = []

for name, loader in [
    ('Departamentos', load_departamentos),
    ('Distritos', load_distritos),
    ('Catastro parcels', load_catastro_parcels),
    ('Indigenous territories', load_indigenous_territories),
]:
    try:
        gdf = loader()
        summary.append({
            'dataset': name,
            'count': len(gdf),
            'crs': gdf.crs.to_string() if gdf.crs else None,
        })
    except Exception as e:
        summary.append({'dataset': name, 'error': str(e)})

# Tiles
try:
    tiles = load_tile_index()
    summary.append({'dataset': 'Tiles (10x10 km)', 'count': len(tiles)})
except Exception as e:
    summary.append({'dataset': 'Tiles', 'error': str(e)})

pd.DataFrame(summary)

In [ ]:
# Plot all datasets on a single map
fig, ax = plt.subplots(1, 1, figsize=(15, 12))

try:
    deptos = load_departamentos()
    deptos.plot(ax=ax, color='lightgray', edgecolor='black', linewidth=1)
    ax.set_title('Paraguay Departamentos')
except Exception as e:
    print(f"Error: {e}")

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('../outputs/figures/departamentos.png', dpi=150)
plt.show()

In [ ]:
# Plot tile grid
fig, ax = plt.subplots(1, 1, figsize=(15, 12))
try:
    tiles = load_tile_index()
    # Build GeoDataFrame from tile_id
    tiles['center_lon'] = tiles['tile_id'].str.split('_').str[0].astype(float)
    tiles['center_lat'] = tiles['tile_id'].str.split('_').str[1].astype(float)
    ax.scatter(tiles['center_lon'], tiles['center_lat'], s=1, c='blue', alpha=0.3)
    ax.set_title(f'Paraguay Tile Grid ({len(tiles)} tiles, 10x10 km)')
    deptos = load_departamentos()
    deptos.plot(ax=ax, color='none', edgecolor='black', linewidth=1)
except Exception as e:
    print(f"Error: {e}")
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('../outputs/figures/tile_grid.png', dpi=150)
plt.show()